# Multi Tasking Model for Hate and Fake

In [2]:
import pandas as pd
import numpy as np
import torch
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the dataset from your .xlsx file
input_file = "<path_to_Dataset>"
df = pd.read_excel(input_file)

# Load embeddings from "embeddings.pt"
all_embeddings = torch.load("<Embeddings in .pt format>").numpy()  # Load with PyTorch and convert to NumPy array

# Prepare the labels
labels_fake = tf.convert_to_tensor(df['Fake_Label'].values) # Replace "Fake_Label" by Fake label Column Header in dataset.
labels_hate = tf.convert_to_tensor(df['Hate_Label'].values) # Replace "Hate_Label" by Hate label Column Header in dataset.

# Convert TensorFlow tensors to NumPy arrays
labels_fake = labels_fake.numpy()
labels_hate = labels_hate.numpy()

# Split the data into train and test sets
train_embeddings, test_embeddings, train_labels_fake, test_labels_fake, train_labels_hate, test_labels_hate = train_test_split(all_embeddings, labels_fake, labels_hate, test_size=0.2, random_state=42)


In [10]:
all_embeddings.shape[1]

768

## Data Preparation for Training and Testing and Custom Dataset Definition

In [3]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt

# Define a custom dataset
class MultiTaskDataset(tf.keras.utils.Sequence):
    def __init__(self, embeddings, labels_fake, labels_hate, batch_size):
        self.embeddings = embeddings
        self.labels_fake = labels_fake
        self.labels_hate = labels_hate
        self.batch_size = batch_size

    def __len__(self):
        return len(self.embeddings) // self.batch_size

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = (idx + 1) * self.batch_size
        return self.embeddings[start:end], self.labels_fake[start:end], self.labels_hate[start:end]

# Create data loaders for training and testing
batch_size = 32
train_dataset = MultiTaskDataset(train_embeddings, train_labels_fake, train_labels_hate, batch_size)
test_dataset = MultiTaskDataset(test_embeddings, test_labels_fake, test_labels_hate, batch_size)

## Multi-Task Model

In [4]:

# Define the multi-task classification model in TensorFlow
def create_multi_task_model(input_dim):
    input_layer = tf.keras.layers.Input(shape=(input_dim,))
    fake_layer = tf.keras.layers.Dense(2048, activation='relu')(input_layer)
    fake_output = tf.keras.layers.Dense(1)(fake_layer)

    hate_layer = tf.keras.layers.Dense(2048, activation='relu')(input_layer)
    hate_output = tf.keras.layers.Dense(1)(hate_layer)

    model = tf.keras.models.Model(inputs=input_layer, outputs=[fake_output, hate_output])

    return model

model = create_multi_task_model(all_embeddings.shape[1])

# Compile the model with hinge loss
model.compile(loss='hinge', optimizer='adam', metrics=['accuracy'])

# Initialize and train the multi-task model
num_epochs = 10

## Metrics Initialization and Tracking

In [5]:
# Lists to store metrics for each epoch
accuracy_fake_list = []
f1_fake_list = []
accuracy_hate_list = []
f1_hate_list = []
epoch_list = []
training_loss_list = []
validation_loss_list = []

max_f1_fake = 0
max_f1_hate = 0

best_epoch_fake = 0
best_epoch_hate = 0

## Metrics Function

In [6]:
# Modify the calculate_metrics function
def calculate_metrics(y_true, y_pred, task_name, epoch):
    y_pred = (np.array(y_pred) >= 0.5).astype(int)
    y_true = y_true[:len(y_pred)]  # Ensure consistent lengths

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"{task_name} Metrics - Epoch {epoch + 1}:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}\n")

    return accuracy, f1  # Return the metrics

# Lists to store metrics for each epoch
training_loss_list = []
validation_loss_list = []

## Training and Testing the Multi-Task Model

In [ ]:
for epoch in range(num_epochs):
    epoch_list.append(epoch + 1)
    epoch_training_loss = []  # List to store training loss for this epoch
    for batch in train_dataset:
        embeddings, labels_fake, labels_hate = batch
        loss = model.train_on_batch(embeddings, [labels_fake, labels_hate])
        epoch_training_loss.append(loss)  # Append training loss for this batch

    # Store average training loss for this epoch
    training_loss_list.append(np.mean(epoch_training_loss))

    # Evaluation on the test set
    fake_preds, hate_preds = [], []
    for batch in test_dataset:
        embeddings, labels_fake, labels_hate = batch
        fake_pred, hate_pred = model.predict(embeddings) # Model.Predict
        fake_preds.extend(fake_pred)
        hate_preds.extend(hate_pred)

    # Ensure consistent lengths of test labels and predictions
    min_length = min(len(test_labels_fake), len(fake_preds))
    test_labels_fake = test_labels_fake[:min_length]
    fake_preds_binary = (np.array(fake_preds)[:min_length] >= 0.5).astype(int)
    test_labels_hate = test_labels_hate[:min_length]
    hate_preds_binary = (np.array(hate_preds)[:min_length] >= 0.5).astype(int)

    # Calculate metrics for fake detection
    accuracy_fake, f1_fake = calculate_metrics(test_labels_fake, fake_preds_binary, "Fake Detection", epoch)
    accuracy_fake_list.append(accuracy_fake)
    f1_fake_list.append(f1_fake)

    # Calculate metrics for hate detection
    accuracy_hate, f1_hate = calculate_metrics(test_labels_hate, hate_preds_binary, "Hate Detection", epoch)
    accuracy_hate_list.append(accuracy_hate)
    f1_hate_list.append(f1_hate)

    # Update max_f1_fake and best_epoch_fake if a new high F1-score is achieved for Fake
    if f1_fake > max_f1_fake:
        max_f1_fake = f1_fake
        best_epoch_fake = epoch + 1

    # Update max_f1_hate and best_epoch_hate if a new high F1-score is achieved for Hate
    if f1_hate > max_f1_hate:
        max_f1_hate = f1_hate
        best_epoch_hate = epoch + 1

    # Evaluate the model on the validation set
    val_loss = model.evaluate(test_dataset)
    validation_loss_list.append(val_loss[0])

## Evaluation and Reporting Metrics

In [ ]:
# Print the highest F1-scores and best epochs for Fake and Hate
print(f"Highest F1-Score for Fake: {max_f1_fake:.4f} at Epoch {best_epoch_fake}")
print(f"Highest F1-Score for Hate: {max_f1_hate:.4f} at Epoch {best_epoch_hate}")

# Display the classification report at the end
print("Classification Report - Fake Detection:")
print(classification_report(test_labels_fake, fake_preds_binary))

print("Classification Report - Hate Detection:")
print(classification_report(test_labels_hate, hate_preds_binary))

# Calculate overall F1-Score for test data
hate_f1 = f1_score(test_labels_hate, hate_preds_binary)
fake_f1 = f1_score(test_labels_fake, fake_preds_binary)

# Calculate the overall F1-Score by averaging the F1-Scores of each task
overall_f1 = (fake_f1 + hate_f1) / 2

print("======================================================")
print("Test Data Classification Report")
print("------------------------------------------------------")
print("Hate F1-Score : ", hate_f1)
print("Fake F1-Score : ", fake_f1)
print(f"Overall F1-Score for Test Data: {overall_f1:.4f}")
print("------------------------------------------------------")

# Plotting the metrics
plt.figure(figsize=(12, 8))

# Plotting Accuracy
plt.subplot(2, 1, 1)
plt.plot(epoch_list, accuracy_fake_list, label='Fake Detection - Accuracy', marker='o')
plt.plot(epoch_list, accuracy_hate_list, label='Hate Detection - Accuracy', marker='o')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plotting F1-score
plt.figure(figsize=(8, 6))
plt.plot(epoch_list, f1_fake_list, label='Fake Detection - F1 Score', marker='o')
plt.plot(epoch_list, f1_hate_list, label='Hate Detection - F1 Score', marker='o')
plt.title('F1 Score Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()